Step 18 — Start the independent Gemma notebook and download the pinned model

In [2]:
import getpass
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
python = Path("/opt/b3it-corrected-venv/bin/python")
project = root / "authors_repo/track-llm-apis"
assert python.is_file(), f"Experiment Python missing: {python}"

code = r'''
import json
import sys
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer

root = Path(sys.argv[1])
model = "google/gemma-3-1b-it"
manifest = json.loads((root / "audit/hub_revisions.json").read_text())
revision = manifest["models"][model]["revision"]

snapshot = Path(snapshot_download(
    repo_id=model,
    revision=revision,
    cache_dir=str(root / "cache/huggingface/hub"),
    allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
))
assert snapshot.name == revision, f"Unexpected snapshot: {snapshot}"

index = snapshot / "model.safetensors.index.json"
if index.is_file():
    shards = sorted(set(json.loads(index.read_text())["weight_map"].values()))
else:
    shards = ["model.safetensors"]

assert shards, "Empty weight shard list"
sizes = {name: (snapshot / name).stat().st_size for name in shards}
assert all(size > 0 for size in sizes.values()), "Empty weight shard"

cfg = AutoConfig.from_pretrained(snapshot, local_files_only=True)
tok = AutoTokenizer.from_pretrained(snapshot, local_files_only=True)
assert "Gemma3ForCausalLM" in (cfg.architectures or []), cfg.architectures
assert tok.encode("Hello", add_special_tokens=False)

record = {
    "model": model,
    "revision": revision,
    "snapshot": str(snapshot),
    "weight_files": sizes,
    "model_type": cfg.model_type,
    "tokenizer_class": type(tok).__name__,
}
path = root / "audit/gemma_snapshot.json"
if path.exists():
    assert json.loads(path.read_text()) == record, "Existing Gemma snapshot record differs."
else:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(record, indent=2) + "\n")
    tmp.replace(path)

print("Revision:", revision)
print("Snapshot:", snapshot)
print("Weights:", round(sum(sizes.values()) / 2**30, 2), "GiB")
print("Step 18 passed: pinned Gemma weights present and tokenizer loaded locally.")
'''

env = {
    **os.environ,
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "HF_HUB_OFFLINE": "0",
    "MPLBACKEND": "Agg",
}
if not env.get("HF_TOKEN"):
    env["HF_TOKEN"] = getpass.getpass("Hugging Face read token: ").strip()

started = time.monotonic()
try:
    subprocess.run([str(python), "-u", "-c", code, str(root)],
                   cwd=project, env=env, check=True)
finally:
    env.pop("HF_TOKEN", None)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

Hugging Face read token:  ········


Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 28851.62it/s]


Revision: dcc83ea841ab6100d6b47a070329e1ba4cf78752
Snapshot: /workspace/B3IT_Figure1_Corrected/cache/huggingface/hub/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752
Weights: 1.86 GiB
Step 18 passed: pinned Gemma weights present and tokenizer loaded locally.
Elapsed: 0.1 minutes


In [ ]:
Step 19 — Discover Gemma’s border inputs.

In [3]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/discover_gemma.py"
log = root / "logs/step19_gemma_discovery.log"

script.write_text(r'''
import json
import sys
from pathlib import Path
from unittest.mock import patch

from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling import bi_phase_1 as phase
from track_llm_apis.util import slugify


def main():
    root = Path(sys.argv[1])
    model = "google/gemma-3-1b-it"
    snap = json.loads((root / "audit/gemma_snapshot.json").read_text())
    prep = json.loads((root / "audit/lmsys_preparation.json").read_text())
    snapshot = Path(snap["snapshot"])
    assert snap["model"] == model and snapshot.is_dir()
    assert snapshot.name == snap["revision"]

    bi = config.sampling.bi
    assert config.data_dir == root / "data"
    assert (bi.phase_1_tokens_per_endpoint, bi.phase_1_queries_per_token) == (20000, 3)
    assert (bi.temperature, bi.models[model]) == (0, 1)
    assert (bi.batch_size, bi.per_batch_input_fraction) == (64, 0.95)

    cache = config.datasets_dir / slugify(prep["dataset"], hash_length=0)
    state = json.loads((cache / "state.json").read_text())
    assert state["_fingerprint"] == prep["dataset_fingerprint"]

    try:
        result = phase.BIPhase1aOutput.load_model(model)
    except FileNotFoundError:
        result = None

    if result is None:
        # Keep the upstream model ID in saved results and load Step 18's exact revision.
        original_init = phase.init_vllm_bi

        class PinnedTokenizer:
            @staticmethod
            def from_pretrained(name):
                assert name == model
                return AutoTokenizer.from_pretrained(str(snapshot), local_files_only=True)

        def pinned_engine(name, device):
            assert name == model
            return original_init(str(snapshot), device)

        print("Starting upstream Gemma discovery on all 20,000 candidates.", flush=True)
        with (
            patch.object(phase, "AutoTokenizer", PinnedTokenizer),
            patch.object(phase, "init_vllm_bi", pinned_engine),
        ):
            phase.phase_1a(device="cuda:0", model=model)
        result = phase.BIPhase1aOutput.load_model(model)
    else:
        print("Found saved Gemma discovery. Verifying every candidate.", flush=True)

    assert result.model_name == model and result.phase == "1a"
    assert result.bi_config == bi.model_dump(mode="json")
    assert len(result.results) == 20000
    assert result.results.keys() == result.prompt_lengths.keys()
    assert all(sum(outputs.values()) == 3 for outputs in result.results.values())

    report = {
        "model": model,
        "revision": snap["revision"],
        "traffic_dataset_fingerprint": prep["dataset_fingerprint"],
        "candidates": len(result.results),
        "candidate_observations": sum(sum(v.values()) for v in result.results.values()),
        "discovered_border_inputs": len(result.get_border_inputs()),
        "upstream_result": str(result._get_path()),
    }
    path = root / "audit/step19_gemma_discovery.json"
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(report, indent=2) + "\n")
    tmp.replace(path)
    print(json.dumps(report, indent=2))
    print("Step 19 passed: 20,000 Gemma candidates and 60,000 target observations verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis",
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 19 exited with code {status}. See {log}")

Starting upstream Gemma discovery on all 20,000 candidates.
2026-11-14 13:10:25,209 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 13:10:25,211 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
2026-11-14 13:10:25,343 - track-llm-apis - INFO - Loaded 4502 traffic prompts
2026-11-14 13:10:25,343 - track-llm-apis - INFO - Processing google/gemma-3-1b-it: 20000 inputs, 3 samples each, max_tokens=1
Enforce eager set, overriding optimization level to -O0
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=10333) 
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00

Step 20 — Calibrate all 867 Gemma border inputs

In [4]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/calibrate_gemma.py"
log = root / "logs/step20_gemma_calibration.log"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path
from unittest.mock import patch

from track_llm_apis.config import config
from track_llm_apis.sampling import bi_phase_1 as phase


def main():
    root = Path(sys.argv[1])
    model = "google/gemma-3-1b-it"
    snap = json.loads((root / "audit/gemma_snapshot.json").read_text())
    discovery = json.loads((root / "audit/step19_gemma_discovery.json").read_text())
    snapshot = Path(snap["snapshot"])
    assert snap["model"] == discovery["model"] == model
    assert snapshot.is_dir() and snapshot.name == snap["revision"] == discovery["revision"]
    assert config.data_dir == root / "data"

    bi = config.sampling.bi
    assert (bi.phase_1_queries_per_bi, bi.max_border_inputs) == (1000, 120)
    assert (bi.temperature, bi.models[model]) == (0, 1)
    assert (bi.batch_size, bi.per_batch_input_fraction) == (64, 0.95)

    phase_a = phase.BIPhase1aOutput.load_model(model)
    discovered = phase_a.get_border_inputs()
    assert phase_a.bi_config == bi.model_dump(mode="json")
    assert len(discovered) == discovery["discovered_border_inputs"]
    assert len(discovered) >= bi.phase_1b_min_bis, "Below the upstream calibration minimum"

    try:
        result = phase.BIPhase1bOutput.load_model(model)
    except FileNotFoundError:
        result = None

    if result is None:
        original_init = phase.init_vllm_bi

        def pinned_engine(name, device):
            assert name == model
            return original_init(str(snapshot), device)

        print(f"Calibrating {len(discovered):,} Gemma inputs × 1,000 observations.", flush=True)
        with patch.object(phase, "init_vllm_bi", pinned_engine):
            phase.phase_1b(device="cuda:0", model=model)
        result = phase.BIPhase1bOutput.load_model(model)
    else:
        print("Found saved Gemma calibration. Verifying every input.", flush=True)

    assert result.model_name == model and result.phase == "1b"
    assert result.bi_config == bi.model_dump(mode="json")
    assert set(result.results) == set(discovered), "Calibration input set differs from discovery"
    assert result.prompt_lengths.keys() == result.results.keys()
    totals = [sum(counts.values()) for counts in result.results.values()]
    assert all(n == 1000 for n in totals), "Incomplete calibration count"

    eligible = result.get_border_inputs()
    selected = eligible[:bi.max_border_inputs]
    assert len(set(selected)) == len(selected)
    record = {
        "model": model,
        "revision": snap["revision"],
        "traffic_dataset_fingerprint": discovery["traffic_dataset_fingerprint"],
        "discovered_inputs": len(discovered),
        "calibration_observations": sum(totals),
        "eligible_after_calibration": len(eligible),
        "selected_count": len(selected),
        "selection_rule": "First 120 returned by phase 1b get_border_inputs()",
        "prompts": selected,
        "prompt_lengths": {p: result.prompt_lengths[p] for p in selected},
        "prompts_sha256": hashlib.sha256(
            json.dumps(selected, ensure_ascii=False).encode("utf-8")
        ).hexdigest(),
        "upstream_result": str(result._get_path()),
    }
    path = root / "results/gemma/selected_probes.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert json.loads(path.read_text()) == record, "Existing probe selection differs"
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(record, ensure_ascii=False, indent=2) + "\n")
        tmp.replace(path)

    print(f"Calibration observations verified: {sum(totals):,}")
    print(f"Eligible after calibration: {len(eligible):,}")
    print(f"Selected probes: {len(selected)}")
    print("Saved:", path)
    if len(selected) < bi.max_border_inputs:
        raise RuntimeError(f"Only {len(selected)} probes qualified. Actual results saved.")
    print("Step 20 passed: full Gemma calibration verified and 120 ordered probes saved.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis",
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 20 exited with code {status}. See {log}")

Calibrating 867 Gemma inputs × 1,000 observations.
2026-11-14 13:15:35,529 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 13:15:35,531 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
2026-11-14 13:15:35,652 - track-llm-apis - INFO - Loaded 4502 traffic prompts
2026-11-14 13:15:35,653 - track-llm-apis - INFO - Processing google/gemma-3-1b-it: 867 inputs, 1000 samples each, max_tokens=1
Enforce eager set, overriding optimization level to -O0
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=11086) 
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?

In [ ]:
Step 21 — Prepare and verify Gemma’s LoRA training data

In [5]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_gemma_lora_data.py"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.tinychange import FinetuningDataModule, TinyChangeConfig
from track_llm_apis.util import patch_chat_template


def main():
    root = Path(sys.argv[1])
    snap = json.loads((root / "audit/gemma_snapshot.json").read_text())
    model = "google/gemma-3-1b-it"
    assert snap["model"] == model
    assert Path(snap["snapshot"]).name == snap["revision"]

    cfg = TinyChangeConfig()
    pool = cfg.finetuning_dataset
    assert cfg.datasets_dir == root / "data/datasets"
    assert len(pool) == max(cfg.finetuning_samples) == 4096
    assert 128 in cfg.finetuning_samples and cfg.seed == 0
    assert cfg.finetuning_batch_size == cfg.finetuning_epochs == 1

    tok = AutoTokenizer.from_pretrained(snap["snapshot"], local_files_only=True)
    tok.name_or_path = model
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    patch_chat_template(tok, config.chat_templates)

    # Independent check of the indices expected from upstream's seeded permutation.
    indices = np.random.RandomState(cfg.seed).permutation(len(pool))[:128].tolist()
    dm = FinetuningDataModule(
        config=cfg, n_samples=128, batch_size=cfg.finetuning_batch_size, tokenizer=tok,
    )
    np.random.seed(cfg.seed)
    dm.prepare_data()
    dm.setup("fit")
    selected = dm.tokenized_dataset
    assert len(selected) == 128

    padded_lengths, target_counts = set(), []
    for position, pool_index in enumerate(indices):
        expected = tok.apply_chat_template(
            pool[pool_index]["conversation"],
            tokenize=True, return_dict=True, return_assistant_tokens_mask=True,
            padding=False, truncation=False,
        )
        row = selected[position]
        ids = row["input_ids"].tolist()
        attention = row["attention_mask"].tolist()
        assistant = row["assistant_tokens_mask"].tolist()
        assert len(ids) == len(attention) == len(assistant), position
        assert row["labels"].tolist() == ids, position
        assert [x for x, keep in zip(ids, attention) if keep] == expected["input_ids"], position
        assert [x for x, keep in zip(assistant, attention) if keep] == expected["assistant_masks"], position
        assert all(mask == 0 for mask, keep in zip(assistant, attention) if not keep), position
        padded_lengths.add(len(ids))
        target_counts.append(sum(assistant[1:]))

    assert len(padded_lengths) == 1, f"Unexpected padded lengths: {padded_lengths}"
    empty = [i for i, n in enumerate(target_counts) if n == 0]
    assert not empty, f"Selected examples with zero trainable assistant tokens: {empty}"

    record = {
        "model": model,
        "revision": snap["revision"],
        "training_pool_size": len(pool),
        "training_pool_fingerprint": pool._fingerprint,
        "seed": cfg.seed,
        "selected_examples": len(selected),
        "selected_pool_indices_in_training_order": indices,
        "indices_sha256": hashlib.sha256(json.dumps(indices).encode()).hexdigest(),
        "padded_tokens_per_example": padded_lengths.pop(),
        "minimum_trainable_assistant_tokens": min(target_counts),
        "data_module_cache": str(dm._get_disk_path()),
    }
    path = root / "audit/step21_gemma_lora_data.json"
    if path.exists():
        assert json.loads(path.read_text()) == record, "Existing Gemma training-data report differs."
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(record, indent=2) + "\n")
        tmp.replace(path)

    print("Verified examples in training order:", len(selected))
    print("Padded tokens per example:", record["padded_tokens_per_example"])
    print("Minimum trainable assistant tokens:", min(target_counts))
    print("Saved:", path)
    print("Step 21 passed: Gemma LoRA training data, order, and masks verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis",
    env={
        **os.environ,
        "TRACKLLM__DATA_DIR": str(root / "data"),
        "HF_HOME": str(root / "cache/huggingface"),
        "HF_HUB_OFFLINE": "1",
        "CUDA_VISIBLE_DEVICES": "",
        "MPLBACKEND": "Agg",
    },
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

W1114 13:56:12.343000 11653 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


[2026-11-14 13:56:12,370] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.


2026-11-14 13:56:15,266 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 13:56:15,267 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
2026-11-14 13:56:16,555 - tinychange - INFO - Preprocessing finetuning dataset...
2026-11-14 13:56:16,693 - tinychange - INFO - Finetuning dataset does not exist at /workspace/B3IT_Figure1_Corrected/data/datasets/d49f919927c095fa/google2fgemma-3-1b-it, preprocessing...
Saving the dataset (1/1 shards): 100%|██████████| 4096/4096 [00:00<00:00, 6557.52 examples/s]


Verified examples in training order: 128
Padded tokens per example: 2470
Minimum trainable assistant tokens: 2
Saved: /workspace/B3IT_Figure1_Corrected/audit/step21_gemma_lora_data.json
Step 21 passed: Gemma LoRA training data, order, and masks verified.
Elapsed: 0.3 minutes


Step 22 — Train and verify Gemma’s LoRA variant.

In [6]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/train_gemma_lora.py"

script.write_text(r'''
import json
import sys
import tempfile
from pathlib import Path
from unittest.mock import patch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis import tinychange as tc
from track_llm_apis.config import config
from track_llm_apis.util import get_model_hash, patch_chat_template


def load_bf16(path):
    return AutoModelForCausalLM.from_pretrained(
        str(path), torch_dtype=torch.bfloat16, local_files_only=True,
    )


def main():
    root = Path(sys.argv[1])
    snap = json.loads((root / "audit/gemma_snapshot.json").read_text())
    data = json.loads((root / "audit/step21_gemma_lora_data.json").read_text())
    assert snap["model"] == data["model"] == "google/gemma-3-1b-it"
    assert snap["revision"] == data["revision"]
    assert data["selected_examples"] == 128 and data["minimum_trainable_assistant_tokens"] > 0
    cache = Path(data["data_module_cache"])
    assert cache.is_dir(), f"Missing Step 21 training cache: {cache}"

    dest = root / "results/gemma/checkpoints/lora_128"
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        saved = json.loads((dest / "COMPLETE.json").read_text())
        assert saved["revision"] == snap["revision"]
        assert saved["training_order_sha256"] == data["indices_sha256"]
        assert saved["optimizer_steps"] == 128
        assert get_model_hash(load_bf16(dest)) == saved["reloaded_hash"]
        print("Step 22 passed: existing Gemma LoRA checkpoint reverified.")
        print("Changed BF16 parameter values:", saved["changed_bf16_parameter_values"])
        print("Checkpoint:", dest)
        return

    assert torch.cuda.is_available(), "Training process cannot see the GPU."
    assert torch.cuda.is_bf16_supported(including_emulation=False)
    cfg = tc.TinyChangeConfig(variants_device="cuda:0")
    assert cfg.finetuning_dataset._fingerprint == data["training_pool_fingerprint"]
    assert cfg.seed == data["seed"] == 0
    assert cfg.finetuning_lr_scale == [1e-6]
    assert cfg.finetuning_batch_size == cfg.finetuning_epochs == 1
    assert (cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout) == (8, 8, 0.0)
    assert cfg.lora_target_modules == ["q_proj", "k_proj", "v_proj", "o_proj"]

    base = load_bf16(snap["snapshot"])
    assert base.device.type == "cpu" and base.dtype == torch.bfloat16
    tokenizer = AutoTokenizer.from_pretrained(snap["snapshot"], local_files_only=True)
    tokenizer.name_or_path = snap["model"]
    patch_chat_template(tokenizer, config.chat_templates)
    change = tc.TinyChange(base, tokenizer, cfg)

    report = {
        "model": snap["model"],
        "revision": snap["revision"],
        "base_hash": change.model_hash,
        "base_dtype": str(base.dtype),
        "base_device": str(base.device),
        "training_device": cfg.variants_device,
        "gpu": torch.cuda.get_device_name(0),
        "training_order_sha256": data["indices_sha256"],
        "padded_tokens_per_example": data["padded_tokens_per_example"],
        "seed": cfg.seed,
        "rank": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": cfg.lora_target_modules,
    }

    # Observe the authors' trainer without changing its batches or optimizer.
    original_fit = tc.pl.Trainer.fit

    def observed_fit(trainer, model, datamodule):
        assert Path(datamodule._get_disk_path()) == cache
        assert not model.model.is_gradient_checkpointing
        trainable = {
            name: str(p.dtype)
            for name, p in model.model.named_parameters() if p.requires_grad
        }
        assert trainable and all("lora_" in name for name in trainable)
        report["trainable_parameters"] = trainable
        report["gradient_checkpointing"] = False
        print("GPU:", report["gpu"], flush=True)
        print("Trainable adapter tensors:", len(trainable), flush=True)
        print("Adapter dtypes:", sorted(set(trainable.values())), flush=True)

        outcome = original_fit(trainer, model=model, datamodule=datamodule)
        assert len(trainer.optimizers) == 1
        optimizer = trainer.optimizers[0]
        assert isinstance(optimizer, torch.optim.SGD)
        assert all(group["lr"] == 1e-6 for group in optimizer.param_groups)
        report["optimizer"] = type(optimizer).__name__
        report["optimizer_steps"] = int(trainer.global_step)
        assert report["optimizer_steps"] == 128
        return outcome

    torch.cuda.reset_peak_memory_stats()
    with patch.object(tc.pl.Trainer, "fit", observed_fit):
        variant = change.finetune(
            lr=cfg.finetuning_lr_scale[0], n_samples=128,
            epochs=cfg.finetuning_epochs, batch_size=cfg.finetuning_batch_size,
            use_lora=True,
        )

    report["peak_torch_allocated_gib"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    report["description"] = variant.description
    report["merged_dtype"] = str(variant.model.dtype)
    report["merged_hash"] = variant.model_hash
    assert variant.model.dtype == torch.bfloat16

    # Publish the checkpoint only after verifying the saved tensors on reload.
    with tempfile.TemporaryDirectory(dir=dest.parent, prefix=".gemma-lora-") as tmp:
        staged = Path(tmp) / "checkpoint"
        variant.model.save_pretrained(staged, safe_serialization=True)
        tokenizer.save_pretrained(staged)
        reloaded = load_bf16(staged)

        expected = variant.model.state_dict()
        actual = reloaded.state_dict()
        assert expected.keys() == actual.keys(), "Checkpoint tensor names differ"
        for name, tensor in expected.items():
            restored = actual[name]
            assert tensor.shape == restored.shape and tensor.dtype == restored.dtype, name
            assert torch.equal(tensor.detach().cpu(), restored), f"Reload mismatch: {name}"

        base_params = dict(base.named_parameters())
        saved_params = dict(reloaded.named_parameters())
        assert base_params.keys() == saved_params.keys()
        changed = 0
        for name, original in base_params.items():
            restored = saved_params[name]
            assert original.dtype == restored.dtype == torch.bfloat16, name
            assert original.shape == restored.shape, name
            changed += int(torch.count_nonzero(original.detach() != restored.detach()).item())

        report["verified_checkpoint_tensors"] = len(actual)
        report["changed_bf16_parameter_values"] = changed
        report["merged_weights_changed"] = changed > 0
        report["reloaded_hash"] = get_model_hash(reloaded)
        (staged / "COMPLETE.json").write_text(json.dumps(report, indent=2) + "\n")
        staged.rename(dest)

    for key in ("optimizer_steps", "merged_dtype", "verified_checkpoint_tensors",
                "changed_bf16_parameter_values", "peak_torch_allocated_gib"):
        print(f"{key}: {report[key]}")
    print("Checkpoint:", dest)
    print("Step 22 passed: Gemma LoRA trained, merged, saved, and reload-verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
log = root / "logs/step22_gemma_lora.log"
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 22 exited with code {status}. See {log}")

2026-11-14 13:59:19,481 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 13:59:19,481 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Seed set to 0
2026-11-14 13:59:39,065 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
/opt/b3it-corrected-venv/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /workspace/B3IT_Figure1_Corrected/scripts/train_gemm ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which lo

In [ ]:
Step 23 — Create Gemma’s Gaussian noise, magnitude-pruning, and random-pruning checkpoints

In [7]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/create_gemma_weight_variants.py"

script.write_text(r'''
import gc
import json
import sys
import tempfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.tinychange import TinyChange, TinyChangeConfig
from track_llm_apis.util import get_model_hash, patch_chat_template


def load_bf16(path):
    return AutoModelForCausalLM.from_pretrained(
        str(path), torch_dtype=torch.bfloat16, local_files_only=True,
    )


def verify_reload(merged, restored, base, name):
    expected = merged.state_dict()
    actual = restored.state_dict()
    assert expected.keys() == actual.keys(), f"{name}: checkpoint tensor names differ"
    for key, tensor in expected.items():
        saved = actual[key]
        assert tensor.shape == saved.shape and tensor.dtype == saved.dtype, key
        assert torch.equal(tensor.detach().cpu(), saved), f"{name}: reload mismatch at {key}"

    base_params = dict(base.named_parameters())
    saved_params = dict(restored.named_parameters())
    assert base_params.keys() == saved_params.keys()
    changed = 0
    for key, original in base_params.items():
        saved = saved_params[key].detach()
        assert saved.shape == original.shape, key
        assert saved.dtype == original.dtype == torch.bfloat16, key
        difference = saved != original.detach()
        count = int(difference.sum().item())
        if name in ("magnitude", "random") and count:
            assert "mlp" in key and key.endswith(".weight"), f"Unexpected pruned tensor: {key}"
            assert torch.count_nonzero(saved[difference]).item() == 0, key
        changed += count
    return len(actual), changed


def main():
    root = Path(sys.argv[1])
    snap = json.loads((root / "audit/gemma_snapshot.json").read_text())
    checkpoints = root / "results/gemma/checkpoints"
    lora = json.loads((checkpoints / "lora_128/COMPLETE.json").read_text())
    assert snap["model"] == lora["model"] == "google/gemma-3-1b-it"
    assert Path(snap["snapshot"]).name == snap["revision"] == lora["revision"]

    base = load_bf16(snap["snapshot"])
    assert base.device.type == "cpu"
    baseline_hash = get_model_hash(base)
    assert baseline_hash == lora["base_hash"], "Baseline differs from Step 22"
    tokenizer = AutoTokenizer.from_pretrained(snap["snapshot"], local_files_only=True)
    tokenizer.name_or_path = snap["model"]
    patch_chat_template(tokenizer, config.chat_templates)

    cfg = TinyChangeConfig(variants_device="cuda:0")
    assert cfg.seed == 0
    settings = {
        "noise": {"type": "random_noise", "scale": 2**-12},
        "magnitude": {"type": "weight_pruning", "method": "magnitude", "scale": 2**-8},
        "random": {"type": "weight_pruning", "method": "random", "scale": 2**-8},
    }

    for name, description in settings.items():
        dest = checkpoints / name
        if dest.exists():
            report = json.loads((dest / "COMPLETE.json").read_text())
            assert report["revision"] == snap["revision"] and report["base_hash"] == baseline_hash
            assert report["description"] == description and report["seed"] == cfg.seed
            assert get_model_hash(load_bf16(dest)) == report["reloaded_hash"]
            print(f"{name}: reused verified checkpoint; "
                  f"{report['changed_bf16_parameter_values']:,} changed values.", flush=True)
            continue

        print(f"Creating {name} from the original Gemma weights...", flush=True)
        change = TinyChange(base, tokenizer, cfg)
        if name == "noise":
            variant = change.random_noise(scale=description["scale"])
        else:
            variant = change.weight_pruning(scale=description["scale"], method=name)

        assert variant.description == description
        assert variant.model.dtype == torch.bfloat16
        assert get_model_hash(base) == baseline_hash, "Original model changed"

        with tempfile.TemporaryDirectory(dir=checkpoints, prefix=f".{name}-") as tmp:
            staged = Path(tmp) / "checkpoint"
            variant.model.save_pretrained(staged, safe_serialization=True)
            tokenizer.save_pretrained(staged)
            restored = load_bf16(staged)
            verified, changed = verify_reload(variant.model, restored, base, name)

            report = {
                "model": snap["model"],
                "revision": snap["revision"],
                "description": description,
                "seed": cfg.seed,
                "rng_policy": "TinyChange initialized independently for each variant",
                "base_device": str(base.device),
                "creation_device": cfg.variants_device,
                "creation_dtype": str(variant.model.dtype),
                "base_hash": baseline_hash,
                "upstream_variant_hash": variant.model_hash,
                "reloaded_hash": get_model_hash(restored),
                "verified_checkpoint_tensors": verified,
                "changed_bf16_parameter_values": changed,
            }
            (staged / "COMPLETE.json").write_text(json.dumps(report, indent=2) + "\n")
            staged.rename(dest)

        print(f"{name}: {changed:,} BF16 values changed; "
              f"{verified} checkpoint tensors verified.", flush=True)
        del restored, variant, change
        gc.collect()
        torch.cuda.empty_cache()

    print("Checkpoint directory:", checkpoints)
    print("Step 23 passed: all three Gemma weight-perturbation checkpoints verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
started = time.monotonic()
log = root / "logs/step23_gemma_weight_variants.log"
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 23 exited with code {status}. See {log}")

`torch_dtype` is deprecated! Use `dtype` instead!
2026-11-14 14:20:41,411 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 14:20:41,412 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
Creating noise from the original Gemma weights...
INFO: Seed set to 0
2026-11-14 14:20:47,476 - lightning.fabric.utilities.seed - INFO - Seed set to 0
2026-11-14 14:20:47,477 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
noise: 828,819,034 BF16 values changed; 341 checkpoint tensors verified.
Creating magnitude from the original Gemma weights...
INFO: Seed set to 0
2026-11-14 14:21:53,377 - lightning.fabric.utilities.seed - INFO - Seed set to 0
2026-11-14 14:21:53,378 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
magnitude: 2,426,112 BF16 values changed; 341 checkpoint tensors verified.
Creating random from the origin

In [ ]:
Step 24 — Verify Gemma’s evaluation inputs and save its manifest

In [8]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_gemma_evaluation.py"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path

from datasets import load_from_disk
from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling.bi_phase_1 import BIPhase1bOutput
from track_llm_apis.util import slugify


def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False).encode()).hexdigest()


def encode(path, texts):
    tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
    return tokenizer(texts, padding=False, truncation=False)["input_ids"]


def main():
    root = Path(sys.argv[1])
    results = root / "results/gemma"
    snap = json.loads((root / "audit/gemma_snapshot.json").read_text())
    selection = json.loads((results / "selected_probes.json").read_text())
    prep = json.loads((root / "audit/lmsys_preparation.json").read_text())
    model = "google/gemma-3-1b-it"

    assert model == snap["model"] == selection["model"]
    assert snap["revision"] == selection["revision"] == Path(snap["snapshot"]).name
    assert config.data_dir == root / "data"
    bi, lp = config.sampling.bi, config.sampling.logprob

    calibration = BIPhase1bOutput.load_model(model, temperature=bi.temperature)
    probes = calibration.get_border_inputs()[:bi.max_border_inputs]
    assert probes == selection["prompts"]
    assert len(probes) == len(set(probes)) == 120
    assert digest(probes) == selection["prompts_sha256"]

    # Check the stored fingerprint and the ordered content used by evaluation.
    cache = (Path(prep["datasets_dir"]) / slugify(prep["dataset"], hash_length=0)).resolve()
    state = json.loads((cache / "state.json").read_text())
    assert state["_fingerprint"] == prep["dataset_fingerprint"] == selection["traffic_dataset_fingerprint"]
    traffic = lp.other_prompts
    expected_files = [(cache / f["filename"]).resolve() for f in state["_data_files"]]
    opened_files = [Path(f["filename"]).resolve() for f in lp.other_prompts_dataset.cache_files]
    assert opened_files == expected_files and expected_files
    assert traffic == [r["conversation"][0]["content"] for r in load_from_disk(str(cache))]
    assert len(traffic) == prep["filtered_conversations"] == 4502
    print("Verified 120 ordered Gemma probes and 4,502 ordered traffic prompts.", flush=True)

    assert (bi.temperature, bi.models[model], lp.batch_size) == (0, 1, 64)
    assert bi.per_batch_input_fraction == 0.95
    assert config.sampling.n_samples == 5000 and bi.reference_samples_per_prompt == 50

    texts = list(dict.fromkeys(probes + traffic))
    baseline_ids = encode(snap["snapshot"], texts)
    conditions = [{
        "id": "unchanged", "model_path": snap["snapshot"],
        "description": {"type": "unchanged"},
    }]
    expected = {
        "lora_128": {"type": "finetune", "lr": 1e-6, "n_samples": 128, "lora": True},
        "noise": {"type": "random_noise", "scale": 2**-12},
        "magnitude": {"type": "weight_pruning", "method": "magnitude", "scale": 2**-8},
        "random": {"type": "weight_pruning", "method": "random", "scale": 2**-8},
    }
    base_hashes = set()
    for name, description in expected.items():
        path = results / "checkpoints" / name
        report = json.loads((path / "COMPLETE.json").read_text())
        assert report["model"] == model and report["revision"] == snap["revision"], name
        assert report["description"] == description, name
        assert report["changed_bf16_parameter_values"] > 0, name
        assert report["reloaded_hash"], name
        assert encode(path, texts) == baseline_ids, f"{name}: tokenizer IDs differ"
        base_hashes.add(report["base_hash"])
        conditions.append({
            "id": name, "model_path": str(path), "description": description,
            "checkpoint_reloaded_hash": report["reloaded_hash"],
        })
        print(f"{name}: checkpoint metadata and token IDs verified.", flush=True)
    assert len(base_hashes) == 1, "The variants have different baseline hashes"

    manifest = {
        "model": model, "revision": snap["revision"],
        "probes": probes, "probes_sha256": digest(probes),
        "traffic_count": len(traffic), "traffic_sha256": digest(traffic),
        "traffic_dataset_fingerprint": prep["dataset_fingerprint"],
        "conditions": conditions, "base_hash": base_hashes.pop(),
        "temperature": bi.temperature, "max_tokens": bi.models[model],
        "batch_size": lp.batch_size, "input_fraction": bi.per_batch_input_fraction,
        "observations_per_probe_per_condition": config.sampling.n_samples,
        "reference_samples_per_prompt": bi.reference_samples_per_prompt,
        "planned_target_observations": len(probes) * len(conditions) * config.sampling.n_samples,
    }
    assert manifest["planned_target_observations"] == 3_000_000
    path = results / "evaluation_manifest.json"
    if path.exists():
        assert json.loads(path.read_text()) == manifest, "Existing Gemma manifest differs"
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
        tmp.replace(path)

    print("Saved:", path)
    print("Step 24 passed: 120 probes × 5 conditions × 5,000 fresh observations planned.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis",
    env={
        **os.environ,
        "TRACKLLM__DATA_DIR": str(root / "data"),
        "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
        "HF_HOME": str(root / "cache/huggingface"),
        "HF_HUB_OFFLINE": "1",
        "CUDA_VISIBLE_DEVICES": "",
        "MPLBACKEND": "Agg",
    },
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

W1114 14:27:04.203000 12346 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'
2026-11-14 14:27:04,682 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 14:27:04,683 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...


Verified 120 ordered Gemma probes and 4,502 ordered traffic prompts.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/gemma/checkpoints/lora_128' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


lora_128: checkpoint metadata and token IDs verified.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/gemma/checkpoints/noise' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


noise: checkpoint metadata and token IDs verified.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/gemma/checkpoints/magnitude' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


magnitude: checkpoint metadata and token IDs verified.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/gemma/checkpoints/random' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


random: checkpoint metadata and token IDs verified.
Saved: /workspace/B3IT_Figure1_Corrected/results/gemma/evaluation_manifest.json
Step 24 passed: 120 probes × 5 conditions × 5,000 fresh observations planned.
Elapsed: 0.2 minutes


Step 25 — Collect the unchanged Gemma baseline.

In [9]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/collect_gemma_baseline.py"

script.write_text(r'''
import hashlib
import json
import random
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling.vllm_sampling import (
    cleanup_vllm, init_vllm, load_model_to_vllm, vllm_inference_bi,
)
from track_llm_apis.util import get_model_hash, patch_chat_template


def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False).encode()).hexdigest()


def verify(record, manifest, manifest_hash):
    assert record["manifest_sha256"] == manifest_hash
    assert record["model"] == manifest["model"] and record["revision"] == manifest["revision"]
    assert record["condition"] == "unchanged"
    assert record["base_hash"] == manifest["base_hash"]
    assert record["inference_dtype"] == "torch.bfloat16" and record["output_tokens"] == 1
    probes = manifest["probes"]
    assert set(record["counts"]) == set(record["prompt_lengths"]) == set(probes)
    for prompt in probes:
        counts = record["counts"][prompt]
        assert all(type(n) is int and n > 0 for n in counts.values()), repr(prompt)
        assert sum(counts.values()) == 5000, repr(prompt)
        assert record["prompt_lengths"][prompt] > 0
    version, state, gauss = record["python_random_state_after_sampling"]
    random.Random().setstate((version, tuple(state), gauss))


def main():
    root = Path(sys.argv[1])
    folder = root / "results/gemma"
    raw = (folder / "evaluation_manifest.json").read_bytes()
    manifest = json.loads(raw)
    manifest_hash = hashlib.sha256(raw).hexdigest()

    assert manifest["model"] == "google/gemma-3-1b-it"
    probes = manifest["probes"]
    assert len(probes) == len(set(probes)) == 120
    assert digest(probes) == manifest["probes_sha256"]
    assert manifest["observations_per_probe_per_condition"] == 5000

    out = folder / "observations/unchanged.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        verify(json.loads(out.read_text()), manifest, manifest_hash)
        print("Step 25 passed: existing Gemma baseline reverified — 600,000 observations.")
        print("Saved:", out)
        return

    assert config.data_dir == root / "data"
    bi, lp = config.sampling.bi, config.sampling.logprob
    traffic = lp.other_prompts
    assert digest(traffic) == manifest["traffic_sha256"]
    assert (lp.batch_size, bi.per_batch_input_fraction) == (64, 0.95)
    assert bi.temperature == manifest["temperature"] == 0
    assert bi.models[manifest["model"]] == manifest["max_tokens"] == 1

    baseline, = [c for c in manifest["conditions"] if c["id"] == "unchanged"]
    path = Path(baseline["model_path"])
    assert path.name == manifest["revision"]
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    patch_chat_template(tokenizer, config.chat_templates)
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=torch.bfloat16, local_files_only=True,
    )
    assert get_model_hash(model) == manifest["base_hash"], "Gemma baseline hash differs"
    model = model.to("cuda:0")

    engine = init_vllm(model, tokenizer, "cuda:0")
    try:
        load_model_to_vllm(engine, model)
        dtype = engine.llm_engine.vllm_config.model_config.dtype
        assert dtype == torch.bfloat16, f"Unexpected engine dtype: {dtype}"

        engine_tokenizer = engine.get_tokenizer()
        for text in dict.fromkeys(probes + traffic):
            assert engine_tokenizer.encode(text) == tokenizer.encode(text), (
                "Engine tokenizer differs from the verified Gemma tokenizer."
            )

        random.seed(config.seed)
        print("Collecting unchanged Gemma: 120 probes × 5,000 observations.", flush=True)
        sampled = vllm_inference_bi(
            llm=engine, inputs=probes, traffic_prompts=traffic,
            n_samples=5000, temperature=bi.temperature,
            input_fraction=bi.per_batch_input_fraction,
            batch_size=lp.batch_size, max_tokens=1,
        )
        record = {
            "model": manifest["model"], "revision": manifest["revision"],
            "condition": "unchanged", "description": baseline["description"],
            "base_hash": manifest["base_hash"], "manifest_sha256": manifest_hash,
            "inference_dtype": str(dtype), "sampling_seed": config.seed,
            "counts": sampled.counts, "prompt_lengths": sampled.prompt_lengths,
            "output_tokens": sampled.output_tokens,
            "python_random_state_after_sampling": random.getstate(),
        }
        tmp = out.with_suffix(".tmp")
        tmp.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
        verify(json.loads(tmp.read_text()), manifest, manifest_hash)
        tmp.replace(out)
    finally:
        cleanup_vllm(engine)

    print("Step 25 passed: 120 unchanged Gemma probes × 5,000 observations verified.")
    print("Total fresh baseline observations: 600,000")
    print("Saved:", out)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
log = root / "logs/step25_gemma_baseline.log"
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 25 exited with code {status}. See {log}")

2026-11-14 14:28:39,947 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 14:28:39,948 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Enforce eager set, overriding optimization level to -O0
The tokenizer you are loading from '/tmp/tmpqlq5tks_' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information.

Step 26 — Collect Gemma’s four changed conditions

In [10]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/collect_gemma_changed.py"

script.write_text(r'''
import hashlib
import json
import random
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling.vllm_sampling import (
    cleanup_vllm, init_vllm, load_model_to_vllm, vllm_inference_bi,
)
from track_llm_apis.util import get_model_hash, patch_chat_template
from collect_gemma_baseline import digest, verify as verify_baseline


def unpack_rng(saved):
    version, state, gauss = saved
    return version, tuple(state), gauss


def verify_changed(record, condition, manifest, manifest_hash, start_hash, lengths):
    assert record["model"] == manifest["model"]
    assert record["revision"] == manifest["revision"]
    assert record["manifest_sha256"] == manifest_hash
    assert record["condition"] == condition["id"]
    assert record["description"] == condition["description"]
    assert record["checkpoint_hash"] == condition["checkpoint_reloaded_hash"]
    assert record["start_rng_sha256"] == start_hash
    assert record["inference_dtype"] == "torch.bfloat16"
    assert record["output_tokens"] == 1
    assert record["prompt_lengths"] == lengths
    assert set(record["counts"]) == set(manifest["probes"])
    for prompt, counts in record["counts"].items():
        assert all(type(n) is int and n > 0 for n in counts.values()), repr(prompt)
        assert sum(counts.values()) == 5000, repr(prompt)
    random.Random().setstate(unpack_rng(record["python_random_state_after_sampling"]))


def main():
    root = Path(sys.argv[1])
    name = sys.argv[2]
    results = root / "results/gemma"
    folder = results / "observations"
    raw = (results / "evaluation_manifest.json").read_bytes()
    manifest = json.loads(raw)
    manifest_hash = hashlib.sha256(raw).hexdigest()

    conditions = manifest["conditions"]
    order = [c["id"] for c in conditions]
    assert order == ["unchanged", "lora_128", "noise", "magnitude", "random"]
    index = order.index(name)
    assert index > 0
    assert manifest["model"] == "google/gemma-3-1b-it"
    probes = manifest["probes"]
    assert len(probes) == len(set(probes)) == 120
    assert digest(probes) == manifest["probes_sha256"]
    assert manifest["observations_per_probe_per_condition"] == 5000

    baseline = json.loads((folder / "unchanged.json").read_text())
    verify_baseline(baseline, manifest, manifest_hash)
    lengths = baseline["prompt_lengths"]
    state = baseline["python_random_state_after_sampling"]

    # Verify every preceding condition before continuing its random sequence.
    for earlier in conditions[1:index]:
        saved = json.loads((folder / f"{earlier['id']}.json").read_text())
        verify_changed(saved, earlier, manifest, manifest_hash, digest(state), lengths)
        state = saved["python_random_state_after_sampling"]

    condition = conditions[index]
    start_hash = digest(state)
    dest = folder / f"{name}.json"
    if dest.exists():
        verify_changed(json.loads(dest.read_text()), condition, manifest,
                       manifest_hash, start_hash, lengths)
        print(f"{name}: reused 600,000 verified observations.", flush=True)
        return

    bi, lp = config.sampling.bi, config.sampling.logprob
    traffic = lp.other_prompts
    assert digest(traffic) == manifest["traffic_sha256"]
    assert lp.batch_size == manifest["batch_size"] == 64
    assert bi.per_batch_input_fraction == manifest["input_fraction"] == 0.95
    assert bi.temperature == manifest["temperature"] == 0
    assert bi.models[manifest["model"]] == manifest["max_tokens"] == 1

    tokenizer = AutoTokenizer.from_pretrained(
        conditions[0]["model_path"], local_files_only=True,
    )
    patch_chat_template(tokenizer, config.chat_templates)
    model = AutoModelForCausalLM.from_pretrained(
        condition["model_path"], torch_dtype=torch.bfloat16, local_files_only=True,
    )
    assert get_model_hash(model) == condition["checkpoint_reloaded_hash"], name
    model = model.to("cuda:0")
    engine = init_vllm(model, tokenizer, "cuda:0")
    try:
        load_model_to_vllm(engine, model)
        dtype = engine.llm_engine.vllm_config.model_config.dtype
        assert dtype == torch.bfloat16
        engine_tokenizer = engine.get_tokenizer()
        for text in dict.fromkeys(probes + traffic):
            assert engine_tokenizer.encode(text) == tokenizer.encode(text), name

        random.setstate(unpack_rng(state))
        print(f"Collecting {name}: 120 probes × 5,000 observations.", flush=True)
        sampled = vllm_inference_bi(
            llm=engine, inputs=probes, traffic_prompts=traffic,
            n_samples=5000, temperature=bi.temperature,
            input_fraction=bi.per_batch_input_fraction,
            batch_size=lp.batch_size, max_tokens=1,
        )
        record = {
            "model": manifest["model"], "revision": manifest["revision"],
            "condition": name, "description": condition["description"],
            "checkpoint_hash": condition["checkpoint_reloaded_hash"],
            "manifest_sha256": manifest_hash, "inference_dtype": str(dtype),
            "start_rng_sha256": start_hash,
            "python_random_state_after_sampling": random.getstate(),
            "counts": sampled.counts, "prompt_lengths": sampled.prompt_lengths,
            "output_tokens": sampled.output_tokens,
        }
        tmp = dest.with_suffix(".tmp")
        tmp.write_text(json.dumps(record, ensure_ascii=False), encoding="utf-8")
        verify_changed(json.loads(tmp.read_text()), condition, manifest,
                       manifest_hash, start_hash, lengths)
        tmp.replace(dest)
        print(f"{name}: 600,000 observations verified and saved.", flush=True)
    finally:
        cleanup_vllm(engine)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
for name in ("lora_128", "noise", "magnitude", "random"):
    log = root / f"logs/step26_gemma_{name}.log"
    with log.open("a", encoding="utf-8") as out:
        with subprocess.Popen(
            ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root), name],
            cwd=root / "authors_repo/track-llm-apis", env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                out.write(line)
                out.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"{name} exited with code {status}. See {log}")

print("Step 26 passed: 3,000,000 fresh Gemma evaluation observations verified across five conditions.")
print("Saved in:", root / "results/gemma/observations")
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

2026-11-14 14:47:12,774 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 14:47:12,776 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Enforce eager set, overriding optimization level to -O0
The tokenizer you are loading from '/tmp/tmpr01sphg6' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information.

In [ ]:
Step 27 — Prepare the Gemma scoring input on CPU

In [11]:
from pathlib import Path
import os
import subprocess
import time

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo" / "track-llm-apis"
python = Path("/opt/b3it-corrected-venv/bin/python")
scripts = root / "scripts"

source = (scripts / "prepare_qwen_scoring_input.py").read_text()
assert '"results/qwen"' in source, "Step 15 script layout differs. Share this output."

gemma_source = (
    source.replace('"results/qwen"', '"results/gemma"')
          .replace(".qwen-merge-", ".gemma-merge-")
          .replace("Step 15 passed: 3,000,000 Qwen",
                   "Step 27 passed: 3,000,000 Gemma")
)

script = scripts / "prepare_gemma_scoring_input.py"
if script.exists():
    assert script.read_text() == gemma_source, f"Existing script differs: {script}"
else:
    script.write_text(gemma_source)

started = time.monotonic()
subprocess.run(
    [str(python), "-u", str(script), str(root)],
    cwd=repo,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "", "MPLBACKEND": "Agg"},
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

W1114 15:52:05.900000 16405 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


[2026-11-14 15:52:06,746] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
unchanged: 120 probes × 5,000 observations verified.
lora_128: 120 probes × 5,000 observations verified.
noise: 120 probes × 5,000 observations verified.
magnitude: 120 probes × 5,000 observations verified.
random: 120 probes × 5,000 observations verified.


2026-11-14 15:52:22,847 - track-llm-apis - INFO - Searching for a .json.gz file in /workspace/B3IT_Figure1_Corrected/results/gemma/.gemma-merge-07czrzqv/figure1_merged...
2026-11-14 15:52:22,848 - track-llm-apis - INFO - Decompressing /workspace/B3IT_Figure1_Corrected/results/gemma/.gemma-merge-07czrzqv/figure1_merged/google2fgemma-3-1b-it.json.gz...
2026-11-14 15:52:22,849 - track-llm-apis - INFO - Loading JSON data from contents of the file...
2026-11-14 15:52:25,662 - track-llm-apis - INFO - Loading compressed output v1...


Scoring input: /workspace/B3IT_Figure1_Corrected/results/gemma/figure1_merged/google2fgemma-3-1b-it.json.gz
Step 27 passed: 3,000,000 Gemma observations reload-verified in upstream format.
Elapsed: 0.7 minutes


Step 28 — Score Gemma with the authors’ B3IT procedure

In [12]:
from pathlib import Path
import json
import re

root = Path("/workspace/B3IT_Figure1_Corrected")
archive = root / "results/gemma/figure1_merged/google2fgemma-3-1b-it.json.gz"
assert archive.is_file(), f"Missing Step 27 output: {archive}"

# Read the exact scoring cell that produced the verified Qwen results.
notebook = Path("/workspace/01_figure1_qwen.ipynb")
cells = json.loads(notebook.read_text(encoding="utf-8"))["cells"]
matches = []
for cell in cells:
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        if "Step 16 passed" in source and "B3IT" in source:
            matches.append(source)

assert len(matches) == 1, (
    f"Found {len(matches)} Step 16 scoring cells in {notebook}. "
    "Save the completed Qwen notebook and check its path."
)

source = matches[0]
for original, replacement in [
    ("Qwen/Qwen2.5-0.5B-Instruct", "google/gemma-3-1b-it"),
    ("Qwen2fQwen2.5-0.5B-Instruct", "google2fgemma-3-1b-it"),
    ("Qwen", "Gemma"),
    ("qwen", "gemma"),
    ("QWEN", "GEMMA"),
]:
    source = source.replace(original, replacement)

source = re.sub(r"(?i)\bstep([ _-]?)16\b", r"Step\g<1>28", source)
assert "0.5B-Instruct" not in source, "A Qwen-specific reference remains."

# Save the complete runnable scoring cell alongside the experiment scripts.
runner = root / "scripts/run_gemma_b3it_step28.py"
if runner.exists():
    assert runner.read_text(encoding="utf-8") == source, f"Existing runner differs: {runner}"
else:
    runner.write_text(source, encoding="utf-8")

print("Scoring input:", archive, flush=True)
print("Saved scoring code:", runner, flush=True)
exec(compile(source, str(runner), "exec"), {"__name__": "__main__", "__file__": str(runner)})

Scoring input: /workspace/B3IT_Figure1_Corrected/results/gemma/figure1_merged/google2fgemma-3-1b-it.json.gz
Saved scoring code: /workspace/B3IT_Figure1_Corrected/scripts/run_gemma_b3it_step28.py
W1114 15:54:01.358000 16557 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'
[2026-11-14 15:54:02,015] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
2026-11-14 15:54:04,326 - track-llm-apis - INFO - Searching for a .json.gz file in /workspace/B3IT_Figure1_Corrected/results/gemma/figure1_merged...
2026-11-14 15:54:04,327 - track-llm-apis - INFO - Decompressing /workspace/B3IT_Figure1_Corrected/results/gemma/figure1_merged/google2fgemma-3-1b-it.json.gz...
2026-11-14 15:54:04,328 - track-llm-apis - INFO - Loading JSON data from contents of the file...
2026-11-14 15:54:07,157 - track-llm-apis - INFO - Loading compressed output v1...
2026-11-14 15:54:20,51

In [ ]:
Step 29 — Export and verify Gemma’s Figure 1 CSVs

In [13]:
from pathlib import Path
from collections import defaultdict
from statistics import fmean
import csv
import json
import math
import re

root = Path("/workspace/B3IT_Figure1_Corrected")
model = "google/gemma-3-1b-it"
stats = root / "results/gemma/figure1_merged/analysis/bi"
assert stats.is_dir() and any(stats.iterdir()), f"Missing Step 28 statistics: {stats}"

# Reuse the export cell from the completed Qwen notebook.
notebook = Path("/workspace/01_figure1_qwen.ipynb")
cells = json.loads(notebook.read_text(encoding="utf-8"))["cells"]
matches = [
    "".join(cell["source"])
    for cell in cells
    if cell["cell_type"] == "code" and "Step 17 passed" in "".join(cell["source"])
]
assert len(matches) == 1, f"Expected one Step 17 export cell, found {len(matches)}."

source = matches[0]
for old, new in [
    ("Qwen/Qwen2.5-0.5B-Instruct", model),
    ("Qwen2fQwen2.5-0.5B-Instruct", "google2fgemma-3-1b-it"),
    ("Qwen", "Gemma"),
    ("qwen", "gemma"),
    ("QWEN", "GEMMA"),
]:
    source = source.replace(old, new)

source = re.sub(r"(?i)\bstep([ _-]?)17\b", r"step\g<1>29", source)
assert "0.5B-Instruct" not in source, "A Qwen-specific reference remains."

runner = root / "scripts/run_gemma_export_step29.py"
if runner.exists():
    assert runner.read_text(encoding="utf-8") == source, f"Existing runner differs: {runner}"
else:
    runner.write_text(source, encoding="utf-8")

exec(compile(source, str(runner), "exec"), {"__name__": "__main__", "__file__": str(runner)})

# Verify the exported files independently of the exporter's success message.
out = root / "results/gemma/figure1"
per_variant = out / "figure1_per_variant.csv"
summary = out / "figure1_summary.csv"

def read_rows(path):
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))

def setting(row):
    return int(row["n_prompts"]), int(row["samples_per_prompt"])

rows = read_rows(per_variant)
means = read_rows(summary)
grid = {(p, s) for p in (1, 3, 5, 10) for s in (1, 3, 10, 20, 50)}
assert len(rows) == 80 and len(means) == 20
assert {row["model"] for row in rows} == {model}
assert {setting(row) for row in rows} == {setting(row) for row in means} == grid

groups = defaultdict(dict)
for row in rows:
    key = setting(row)
    variant = json.dumps(json.loads(row["variant"]), sort_keys=True)
    auc = float(row["roc_auc"])
    assert variant not in groups[key], f"Duplicate perturbation at {key}"
    assert 0 <= auc <= 1, f"Invalid AUC at {key}"
    groups[key][variant] = auc

variants = set(groups[(1, 1)])
assert len(variants) == 4
for row in means:
    key = setting(row)
    assert set(groups[key]) == variants, f"Perturbations differ at {key}"
    expected = fmean(groups[key].values())
    assert math.isclose(float(row["mean_roc_auc"]), expected, abs_tol=1e-9), key

print(f"Per-variant CSV: {per_variant}")
print(f"Summary CSV: {summary}")
print("Step 29 verified: 80 Gemma AUC rows and 20 equal-weight four-variant means.")

W1114 15:58:00.460000 16772 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


[2026-11-14 15:58:01,130] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.


2026-11-14 15:58:03,380 - track-llm-apis - INFO - [DataSource.BI] Loading results from /workspace/B3IT_Figure1_Corrected/results/gemma/figure1_merged...


80 rows saved and reread: /workspace/B3IT_Figure1_Corrected/results/gemma/figure1/figure1_per_variant.csv
20 rows saved and reread: /workspace/B3IT_Figure1_Corrected/results/gemma/figure1/figure1_summary.csv
1 prompt(s) × 1 detection sample(s): mean AUC 0.702075, annual cost $0.1577
10 prompt(s) × 50 detection sample(s): mean AUC 0.929945, annual cost $78.8400
step 29 passed: 80 per-variant rows and 20 four-variant means verified.
Elapsed: 0.2 minutes
Per-variant CSV: /workspace/B3IT_Figure1_Corrected/results/gemma/figure1/figure1_per_variant.csv
Summary CSV: /workspace/B3IT_Figure1_Corrected/results/gemma/figure1/figure1_summary.csv
Step 29 verified: 80 Gemma AUC rows and 20 equal-weight four-variant means.
